II.Phân tích missing, quyết định strategy, demo encoding — viết giải thích vào notebook

1. Xem tổng quan các cột

In [ ]:
#import library
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
import os



#hàm load data set để handle error và tái sử dụng
def load_dataset(dataset_path):
    """
    Kiểm tra và lấy dữ liệu từ file csv
    Args:
        dataset_path (str): đường dẫn file csv
    Returns:
        pandas.dataframe: bảng dữ liệu đã được đọc
    """
    if not os.path.exists(dataset_path):
        print("Không tìm thấy file!")
        return None

    try:
        df = pd.read_csv(dataset_path)
        print("Đã tải dữ liệu thành công!")
        return df
    except Exception as e:
        print(f"Đã xảy ra lỗi trong quá trình tải: {e}")
        return None


#define dataset path
dataset_path = "../data/dataset.csv"

#load data
df = load_dataset(dataset_path=dataset_path)

#Hiển thị tổng quan về các thông tin của dataset: số hàng, số cột, kiểu dữ liệu, số giá trị non-null mỗi cột, ...
print("Tổng quan cấu trúc dataset:")

#Thông tin số dòng số cột
print(f"- Dataset có {df.shape[0]} dòng và {df.shape[1]} cột.")
#Thông tin kiểu dữ liệu từng cột, số giá trị non-null
print("- Các thông tin cơ bản:")
df.info()
#Thông tin về trung bình, độ lệch chuẩn, tứ phân vị, giá trị nhỏ nhất lớn nhất
print("- Các thông tin thống kê numerical:")
display(df.describe())
print("- Các thông tin thống kê categorical:")
display(df.describe(include='object'))

In [ ]:
#Hiển thị tổng quan về các thông tin của dataset: số hàng, số cột, kiểu dữ liệu, số giá trị non-null mỗi cột, ...
print("Tổng quan cấu trúc dataset:")

#Thông tin số dòng số cột
print(f"- Dataset có {df.shape[0]} dòng và {df.shape[1]} cột.")

In [ ]:
#Thông tin kiểu dữ liệu từng cột, số giá trị non-null
print("- Các thông tin cơ bản:")
df.info()

In [ ]:
#Thông tin về trung bình, độ lệch chuẩn, tứ phân vị, giá trị nhỏ nhất lớn nhất
print("- Các thông tin thống kê numerical:")
display(df.describe())
print("- Các thông tin thống kê categorical:")
display(df.describe(include='object'))

In [ ]:
# hàng dữ liệu đầu tiên
df.head()

In [ ]:
#ten các cột
df.columns.tolist()

In [ ]:
###1.Thống kê missing values
missing_summary = pd.DataFrame({
    'column': df.columns,
    'missing_count': df.isnull().sum(),
    'missing_percent': df.isnull().mean() * 100
}).sort_values(by='missing_percent', ascending=False)

print(missing_summary)

In [ ]:
###2. Visualize missing
###a. Bar chart tỷ lệ missing
plt.figure(figsize=(10,5))
sns.barplot(x='missing_percent', y='column', data=missing_summary)
plt.title('Tỷ lệ missing theo từng cột (%)')
plt.xlabel('Percent')
plt.ylabel('Column')
plt.show()

In [ ]:
###b. Heatmap missing pattern
plt.figure(figsize=(12,6))
sns.heatmap(df.isnull(), cbar=False)
plt.title('Heatmap Missing Values')
plt.xlabel('Columns')
plt.ylabel('Rows')
plt.show()

In [ ]:
###3. Kiểm tra nhanh các trường hợp quan trọng
# tổng số missing toàn dataset
total_missing = df.isnull().sum().sum()
print("Tổng số giá trị thiếu:", total_missing)

# cột nào > 50% missing
high_missing_cols = missing_summary[missing_summary['missing_percent'] > 50]
print("Cột có >50% missing:\n", high_missing_cols)

5.Nhận Xét
- Dataset hầu như không có missing values đáng kể / hoặc có một số cột bị thiếu nhẹ.
- Không thấy pattern missing rõ ràng giữa các dòng.
- Không có cột nào vượt quá 50% missing nên chưa cần loại bỏ.
- Tuy nhiên khi deploy thực tế, dữ liệu bệnh nhân mới có thể thiếu các cột như ca hoặc thal, cần chuẩn bị pipeline xử lý.

2. Quyết định chiến lược

Chọn Strategy cho từng cột
- mean: age, trestbps, thalach
- median: chol, oldpeak
- mode: sex, cp, fbs, restecg, exang, slope, thal, ca

In [ ]:
###1. Vẽ biểu đồ cho cột số (numerical)
num_cols = df.select_dtypes(include=['int64', 'float64']).columns

for col in num_cols:
    plt.figure(figsize=(10,4))

    plt.subplot(1,2,1)
    sns.histplot(df[col], kde=True)
    plt.title(f'Histogram - {col}')

    plt.subplot(1,2,2)
    sns.boxplot(x=df[col])
    plt.title(f'Boxplot - {col}')

    plt.show()

In [ ]:
###2. Xem phân phối cột danh mục (categorical)
cat_cols = df.select_dtypes(include=['object', 'category']).columns

for col in cat_cols:
    print(f"\n{col}")
    print(df[col].value_counts())

In [ ]:
###vì cp, thal, slope, restecg thường là categorical dạng số nên ta ép kiễu
cat_cols = ['sex','cp','fbs','restecg','exang','slope','ca','thal','num']

- Dựa trên histogram và boxplot, một số cột số như chol và oldpeak xuất hiện outlier rõ ràng, có thể làm lệch giá trị trung bình.
- Các cột như age, thalach có phân phối tương đối ổn định, ít bị ảnh hưởng bởi outlier.
- Các cột categorical như cp, thal, slope không có thứ tự tự nhiên và có nhiều nhóm giá trị khác nhau.
- Các cột nhị phân như sex, fbs, exang đã được mã hóa dạng 0/1 nên không cần xử lý thêm.

Bảng Quyết Định

| Cột      | Strategy                     | Lý do                                              |
| -------- | ---------------------------- | -------------------------------------------------- |
| age      | Giữ nguyên, scale            | Không có outlier rõ ràng, phân phối khá đều        |
| trestbps | Dùng median nếu có missing   | Boxplot có một vài outlier → mean dễ bị lệch       |
| chol     | Dùng median                  | Boxplot cho thấy outlier lớn (~500+) kéo lệch mean |
| thalach  | Scale                        | Phân phối ổn, không outlier nghiêm trọng           |
| oldpeak  | Median + scale               | Có lệch phải và outlier                            |
| ca       | Có thể giữ nguyên hoặc scale | Giá trị nhỏ (0–3), có thứ tự                       |
| sex      | Giữ nguyên (0/1)             | Đã là nhị phân                                     |
| fbs      | Giữ nguyên (0/1)             | Không cần encode                                   |
| exang    | Giữ nguyên (0/1)             | Không cần encode                                   |
| cp       | One-hot encoding             | Là categorical không có thứ tự                     |
| restecg  | One-hot encoding             | Không có thứ tự                                    |
| slope    | One-hot encoding             | Không có thứ tự                                    |
| thal     | One-hot encoding             | Nhiều nhóm khác nhau                               |
| target   | Không xử lý                  | Biến mục tiêu                                      |

- Các cột số có outlier hoặc phân phối lệch được xử lý bằng median để giảm ảnh hưởng của giá trị cực đoan.
- Các cột số còn lại được scale để đưa về cùng thang đo, giúp mô hình học tốt hơn.
- Các cột categorical không có thứ tự được chuyển sang one-hot encoding để tránh mô hình hiểu sai quan hệ giữa các giá trị.
- Các cột nhị phân được giữ nguyên vì đã ở dạng phù hợp.

3.Demo kiểm tra strategy

In [ ]:
###1.Tạo X_train
# fix tên cột nếu có khoảng trắng
df.columns = df.columns.str.strip()

# target đúng
target_col = 'num'

# tách X, y
X = df.drop(target_col, axis=1)
y = df[target_col]

# bỏ cột không cần
X = X.drop(['id', 'dataset'], axis=1)

# chia train/test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(" Đã tạo X_train")

In [ ]:
###2.Demo subset
subset = X_train.head(10).copy()

print("=== TRƯỚC ===")
print(subset)

print("\nNaN ban đầu:")
print(subset.isnull().sum())

In [ ]:
###3.Áp dụng SimpleImputer
# chia nhóm cột
mean_cols = ['age', 'trestbps', 'thalch']
median_cols = ['chol', 'oldpeak']
mode_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal', 'ca']

# mean
imputer_mean = SimpleImputer(strategy='mean')
subset[mean_cols] = imputer_mean.fit_transform(subset[mean_cols])

# median
imputer_median = SimpleImputer(strategy='median')
subset[median_cols] = imputer_median.fit_transform(subset[median_cols])

# mode
imputer_mode = SimpleImputer(strategy='most_frequent')
subset[mode_cols] = imputer_mode.fit_transform(subset[mode_cols])

In [ ]:
###4.Kiểm tra sau khi xử lý
print("\n=== SAU ===")
print(subset)

print("\nNaN còn lại:")
print(subset.isnull().sum())

In [ ]:
###5.Xem giá trị điền
print("\nGiá trị đã dùng:")

print("Mean:", dict(zip(mean_cols, imputer_mean.statistics_)))
print("Median:", dict(zip(median_cols, imputer_median.statistics_)))
print("Mode:", dict(zip(mode_cols, imputer_mode.statistics_)))

4.Demo OneHotEncoder

In [ ]:
###1.Lấy subset + cột cp
subset_cp = X_train[['cp']].head(10).copy()

print("=== TRƯỚC OHE ===")
print(subset_cp)

In [ ]:
###2. Áp dụng OneHotEncoder
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False)

cp_encoded = ohe.fit_transform(subset_cp)

# lấy tên cột mới
cp_columns = ohe.get_feature_names_out(['cp'])

# chuyển về DataFrame cho dễ nhìn
import pandas as pd
cp_df = pd.DataFrame(cp_encoded, columns=cp_columns)

print("\n=== SAU OHE ===")
print(cp_df)

In [ ]:
result = pd.concat([subset_cp.reset_index(drop=True), cp_df], axis=1)
print("\n=== SO SÁNH ===")
print(result)

Kết quả sau khi áp dụng OneHotEncoder cho cột cp. Sau khi áp dụng OneHotEncoder cho cột cp, dữ liệu được chuyển từ dạng categorical (chuỗi) sang dạng số với nhiều cột nhị phân.

Cụ thể, cột cp có 4 giá trị khác nhau nên sau khi encode tạo ra 4 cột mới:
- cp_typical angina
- cp_asymptomatic
- cp_non-anginal
- cp_atypical angina

Mỗi dòng dữ liệu chỉ có một giá trị bằng 1, các cột còn lại bằng 0

In [ ]:
###3.Kiểm tra logic OHE
print("\nTổng mỗi hàng (phải = 1):")
print(cp_df.sum(axis=1))

Khi kiểm tra tổng các giá trị trên mỗi hàng sau khi encode, kết quả luôn bằng 1.
Điều này xác nhận rằng:
- Mỗi bệnh nhân chỉ thuộc một loại đau ngực duy nhất
- Không có trường hợp dữ liệu bị encode sai (ví dụ nhiều hơn 1 giá trị bằng 1)

Số lượng cột sau khi áp dụng OneHotEncoder

Các cột categorical trong dataset gồm:
- sex: 2 giá trị → 2 cột
- cp: 4 giá trị → 4 cột
- fbs: 2 giá trị → 2 cột
- restecg: 3 giá trị → 3 cột
- exang: 2 giá trị → 2 cột
- slope: 3 giá trị → 3 cột
- thal: 3 giá trị → 3 cột

 Tổng số cột sau OneHotEncoder: 19 cột

Tổng số cột input sau khi encode
- Số cột ban đầu: 13 cột
- Số cột categorical được thay thế: 7 cột
- Số cột mới sau encode: 19 cột

Tổng số cột sau xử lý: 13 - 7 + 19 = 25 cột

Kết luận : OneHotEncoder là phương pháp phù hợp cho các cột nominal (không có thứ tự). Giúp model học đúng bản chất dữ liệu . Tránh sai lệch do hiểu nhầm thứ tự giữa các giá trị . Tuy làm tăng số chiều dữ liệu, nhưng đảm bảo tính chính xác và hiệu quả của mô hình